# ML-09 — Validation and Research Claim Audit

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/tabeeljohn2/flyrank-ml-internship/blob/main/work/notebooks/w06_validation_audit.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [1]:
import os, subprocess
import pandas as pd, numpy as np
if not os.path.exists("data/raw/content_refresh_anonymized.csv"):
    if not os.path.exists("/content/flyrank-ml-internship"):
        subprocess.run(["git","clone","https://github.com/tabeeljohn2/flyrank-ml-internship.git","/content/flyrank-ml-internship"])
    os.chdir("/content/flyrank-ml-internship")
df = pd.read_csv("data/raw/content_refresh_anonymized.csv")

from sklearn.model_selection import GroupShuffleSplit, train_test_split
from sklearn.metrics import roc_auc_score
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier

d = df[df["impressions_prev_30d"] >= 50].copy()
d["ratio"] = d["impressions_last_30d"] / d["impressions_prev_30d"]
d["declined"] = (d["ratio"] < 0.7).astype(int)
NUM = ["days_since_last_update","content_age_days","word_count","search_volume","competition","cpc"]
CAT = ["content_type","main_intent"]
FEATURES = NUM + CAT
pre = ColumnTransformer([
    ("num", Pipeline([("imp", SimpleImputer(strategy="median")), ("sc", StandardScaler())]), NUM),
    ("cat", Pipeline([("imp", SimpleImputer(strategy="most_frequent")),
                      ("oh", OneHotEncoder(handle_unknown="ignore"))]), CAT)])
models = {
 "Logistic Regression": Pipeline([("pre", pre), ("m", LogisticRegression(max_iter=1000, class_weight="balanced"))]),
 "Random Forest": Pipeline([("pre", pre), ("m", RandomForestClassifier(n_estimators=200, max_depth=8, min_samples_leaf=50, random_state=42, n_jobs=-1))]),
}
print(d.shape, "base rate", round(d["declined"].mean(), 3))

(20249, 46) base rate 0.521


In [2]:
!pip -q install pypdf
from pypdf import PdfReader
path = "docs/flyrank-seo-research-march-2026.pdf"
if not os.path.exists(path):
    !wget -q -O docs/flyrank-seo-research-march-2026.pdf https://raw.githubusercontent.com/flyrank-bih/flyrank-ml-internship-starter/main/docs/flyrank-seo-research-march-2026.pdf
r = PdfReader(path)
print(len(r.pages), "pages")
for i, p in enumerate(r.pages):
    print("\n--- page", i+1); print(p.extract_text())

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 395.5/395.5 kB 10.7 MB/s eta 0:00:00
36 pages

--- page 1
DATA REPORT · MARCH 2026
THE STATE OF
AI-DRIVEN SEO
IN NUMBERS.
We analyzed 341,701 content pieces across 57 brands, using 
direct portfolio comparisons as the main evidence base and 
an exploratory ML appendix on 61,790 active content records. 
The goal is a public-safe study that a reader can act on without 
FlyRank tooling.
flyrank.com
FlyRank — The State of AI-Driven SEO, March 2026 1

--- page 2
FlyRank ABSTRACT
About This Study
This paper examines 341,701 content pieces across 57 brands, spanning 469,879,632 search impressions, 1,514,819 
clicks,1,635,404 sessions, and 17,344 AI sessions.
Rather than reproducing dashboard screenshots or internal table views, we tested portfolio-level questions about 
content age, freshness, position, traffic mix, depth, and AI visibility. Direct aggregate comparisons lead the paper. 
Machine-learning pages appear later as exploratory appendix mat

In [3]:
!cat skills/hunting-leakage-and-validating/SKILL.md

---
name: hunting-leakage-and-validating
description: Finds label leakage and designs honest validation — leakage taxonomy, grouped and time-aware splits, base rates, and the attack-your-own-model checklist. Use before trusting any metric, when a score looks too good, or when features and labels share time windows or origins.
---

# Hunting leakage and validating

The sneakiest failure in applied ML: the model quietly reads the answer during training. Scores
look amazing; the work is worthless. Your job is to attack your own model before anyone else can.

## The leakage taxonomy — three ways answers sneak in

**1. Label-derived features.** The label was computed FROM a column, and that column (or its
sibling) is in the features. Symptom: one feature towers over all others, and the score is
near-perfect. Test: train once WITH the suspect, once WITHOUT — a collapse from ~1.0 to ~0.7
is the confession.

**2. Future/overlapping windows.** A feature summed over a window that CONTAINS the la

## 1. Two paper findings + my methodology questions

*Pick two findings from the FlyRank research paper. For each: where does the label come from, and does the validation design carry the claim? Constructive tone.*

Finding: (one sentence from the paper, in your own words, with its page or section)

Question I would ask: Where does the label or outcome come from? Is it observed, or defined by a rule? Does the validation design (split, window, comparison group) support the claim as stated?

Why it matters: (one sentence on what would change if the answer were different)

Fair to the authors: (one sentence on what the paper already discloses or does well)

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 2. My model under an honest split (before/after)

*Re-run your Week-5 model under a grouped or time-aware split. Show both numbers.*

In [4]:
def run(tr, te):
    bb = 0.6*te["days_since_last_update"].rank(pct=True) + 0.4*te["impressions_90d"].rank(pct=True)
    row = {"Week-4 rule": roc_auc_score(te["declined"], bb)}
    for n, m in models.items():
        m.fit(tr[FEATURES], tr["declined"])
        row[n] = roc_auc_score(te["declined"], m.predict_proba(te[FEATURES])[:,1])
    return row

rows = []
for seed in range(10):
    tr, te = train_test_split(d, test_size=0.2, random_state=seed)
    r = run(tr, te); r["split"] = "BEFORE: random rows"
    r["shared_clients"] = len(set(tr.client_id) & set(te.client_id)); rows.append(r)
    a, c = next(GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=seed).split(d, d["declined"], groups=d["client_id"]))
    r = run(d.iloc[a], d.iloc[c]); r["split"] = "AFTER: grouped by client"
    r["shared_clients"] = 0; rows.append(r)

ba = pd.DataFrame(rows)
ba.groupby("split")[["Week-4 rule","Logistic Regression","Random Forest"]].agg(["mean","std"]).round(3)

Week-4 rule        Logistic Regression         \
                                mean    std                mean    std   
split                                                                    
AFTER: grouped by client       0.420  0.048               0.583  0.125   
BEFORE: random rows            0.437  0.012               0.629  0.007   

                         Random Forest         
                                  mean    std  
split                                          
AFTER: grouped by client         0.585  0.102  
BEFORE: random rows              0.691  0.011

Before: random 80/20 row split, 10 seeds. Mean AUC was 0.437 (rule), 0.629 (Logistic Regression) and 0.691 (Random Forest), with small spread (std 0.007 to 0.012). After: split by client, same 10 seeds. Mean AUC was 0.420, 0.583 and 0.585, with std 0.048, 0.125 and 0.102. In the random split, pages from the same client appear in both train and test. The unfitted rule barely changed (0.437 to 0.420), while the fitted models dropped (Logistic Regression by 0.046, Random Forest by 0.106). This is consistent with the models using client-specific patterns that do not carry over to unseen clients; I did not isolate the mechanism. Random Forest's apparent edge over Logistic Regression under the random split (0.691 vs 0.629) is gone under the grouped split (0.585 vs 0.583). I report the grouped numbers, because they match the question "does this work on a client the model has not seen?"

## 3. Leakage audit

*The same hunt from Week 3, on your final feature set.*

| Feature | Known at snapshot? | Overlaps label window? | Verdict |
|---|---|---|---|
| days_since_last_update, content_age_days | yes | no | used |
| word_count, search_volume, competition, cpc | yes | no | used |
| content_type, main_intent | yes | no | used |
| impressions_last_30d, impressions_prev_30d, clicks_last_30d, sessions_last_30d | no | yes (they build the label) | excluded |
| trend_direction, trend_pct | no | yes (computed from the label windows) | excluded |
| impressions_90d, clicks_90d, ctr, avg_position, engagement_rate | no | yes (90-day windows include the last 30 days) | excluded |

The Week-4 baseline used impressions_90d, so its AUC of 0.36 is likely partly mechanical.

3b. Label-shuffle test

In [5]:
a, c = next(GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=42).split(d, d["declined"], groups=d["client_id"]))
tr, te = d.iloc[a].copy(), d.iloc[c]
tr["declined"] = np.random.RandomState(0).permutation(tr["declined"].values)
m = models["Logistic Regression"].fit(tr[FEATURES], tr["declined"])
print("shuffled-label AUC:", round(roc_auc_score(te["declined"], m.predict_proba(te[FEATURES])[:,1]), 3))

shuffled-label AUC: 0.415


With labels shuffled in training, the AUC on held-out clients was 0.415 (one grouped split, seed 42). It is not above chance, so this test gives no sign of a pipeline leak. The distance from 0.5 is consistent with the noise seen elsewhere (split-to-split std of about 0.1 with 30 clients). I ran it once; repeating it over several seeds would be stronger.

3c. Drop-the-top-feature test

In [6]:
F2 = [f for f in FEATURES if f != "content_age_days"]
a, c = next(GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=42).split(d, d["declined"], groups=d["client_id"]))
tr, te = d.iloc[a], d.iloc[c]
pre2 = ColumnTransformer([
    ("num", Pipeline([("imp", SimpleImputer(strategy="median")), ("sc", StandardScaler())]), [f for f in NUM if f != "content_age_days"]),
    ("cat", Pipeline([("imp", SimpleImputer(strategy="most_frequent")), ("oh", OneHotEncoder(handle_unknown="ignore"))]), CAT)])
m2 = Pipeline([("pre", pre2), ("m", LogisticRegression(max_iter=1000, class_weight="balanced"))]).fit(tr[F2], tr["declined"])
print("AUC without content_age_days:", round(roc_auc_score(te["declined"], m2.predict_proba(te[F2])[:,1]), 3))

AUC without content_age_days: 0.475


Without content_age_days, the AUC on the same grouped split fell to 0.475, from 0.585 with it. Observed: the signal in this model appears to rest on one feature, and the other features added no measurable lift here.

3d. Why does the AUC vary so much by split?

In [7]:
sizes = d["client_id"].value_counts()
print("clients:", len(sizes), "| largest client share:", round(sizes.iloc[0]/len(d), 3))
print(sizes.head(5))

clients: 30 | largest client share: 0.31
client_id
client_19581e27de    6276
client_6208ef0f77    3512
client_4e07408562    2254
client_3fdba35f04    1975
client_7f2253d7e2    1003
Name: count, dtype: int64


There are 30 clients. The largest holds 31% of pages (6,276 of 20,249), and the top five hold about 74%. A 20% client holdout therefore contains only a few clients, and which clients land in the test set can change the AUC a lot (0.455 to 0.876 for Logistic Regression across 10 splits). This is directional; I did not test it formally.

Real failure examples

In [8]:
a, c = next(GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=42).split(d, d["declined"], groups=d["client_id"]))
tr, te = d.iloc[a], d.iloc[c].copy()
m = models["Logistic Regression"].fit(tr[FEATURES], tr["declined"])
te["score"] = m.predict_proba(te[FEATURES])[:,1]
cols = ["content_id","score","declined","content_age_days","days_since_last_update","content_type","impressions_prev_30d","impressions_last_30d"]
print("Confident wrong (predicted decline, did not):"); print(te[te.declined==0].nlargest(3, "score")[cols])
print("Missed declines (low score, did decline):");     print(te[te.declined==1].nsmallest(3, "score")[cols])

Confident wrong (predicted decline, did not):
                 content_id     score  declined  content_age_days  \
14430  content_a278bb5d581a  0.650964         0               280   
9441   content_1b8802e11084  0.647926         0               118   
15216  content_16bd4fc24d87  0.644889         0                90   

       days_since_last_update     content_type  impressions_prev_30d  \
14430                      20  keyword article                   833   
9441                       20  keyword article                    54   
15216                      20  keyword article                    66   

       impressions_last_30d  
14430                   798  
9441                     43  
15216                   200  
Missed declines (low score, did decline):
               content_id     score  declined  content_age_days  \
237  content_9d94766abf35  0.202356         1               557   
288  content_1da659104d64  0.202356         1               557   
846  content_d0ca9b3a8a26

Confident wrong: three keyword articles (ages 280, 118 and 90 days; 20 days since update; score about 0.65). Their impressions went 833 to 798 (ratio 0.96), 54 to 43 (0.80) and 66 to 200 (3.0). None was a sharp decline, and two had small bases (54 and 66 impressions), where ratios are noisy. Missed declines: three keyword articles, all aged 557 days, all with the identical score 0.202. Impressions went 190 to 115 (0.61), 55 to 6 (0.11) and 284 to 150 (0.53). The identical scores show the model scores mostly by age. Observed pattern: failures involve small impression bases and ratios near my chosen 0.7 threshold. I did not test why.

| Earlier claim | Evidence (measured) | Rewritten claim |
|---|---|---|
| The model beats the Week-4 rule. | Over 10 grouped client splits, mean AUC was 0.583 (Logistic Regression) and 0.585 (Random Forest) vs 0.420 for the rule. Logistic Regression ranged from 0.455 to 0.876 across splits. | Observed on 10 grouped splits, the models had a higher mean AUC than the rule. The result varied widely by client split, so it is directional only. |
| Staleness drives decline. | Signal 1 verdict was MIXED. Permutation importance for days_since_last_update was -0.003. | Staleness was not observed to separate declining pages in this sample. |
| Page age matters. | Decline rate by age bucket: 0.603, 0.656, 0.593, 0.523, 0.278. content_age_days was the only feature with importance (AUC drop 0.099). | Observed: newer pages declined more often. One feature carries the model, and I did not test why. |
| The model finds the pages worth refreshing. | precision@322 was 0.705 vs a base rate of 0.457 on one split. | As decision-support, the top-ranked pages declined more often than average in this sample. This does not show that a refresh would help, because the label is a drop in impressions and not the effect of a refresh. |
| The Week-4 rule failed. | Test AUC 0.360, and below 0.5 on all 10 splits. impressions_90d overlaps the label window. | The rule scored below chance in this sample. This is likely partly mechanical, because its visibility input overlaps the label window. |
| Random-split results show how the model performs. | Random row split: 0.629 (LR) and 0.691 (RF) vs 0.583 and 0.585 grouped. | Observed: a random row split gave higher AUCs than a client-grouped split for both models, so I report the grouped numbers. |
| The model combines several signals. | One grouped split: AUC 0.475 without content_age_days vs 0.585 with it. | Observed: the model's signal appears to rest on one feature. |

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Self-check

Before you submit, confirm each line honestly:

- [✅] Every section above is filled — markdown thinking AND the code that backs it
- [✅] The notebook runs top to bottom with no errors (Runtime → Run all)
- [✅] No client names, URLs, or private queries anywhere
- [✅] My claims use careful words: observed, measured, directional, decision-support
- [✅] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.